Import Libraries

In [1]:
import gymnasium as gym
import numpy as np
import random
import ale_py
from gymnasium.envs.registration import make, pprint_registry, register, registry, spec
from IPython.display import clear_output
import time

import torch
import torch.nn as nn
import torch.optim as optim

Initialize Gym Variables

In [20]:
# gym.register_envs(ale_py)
# env_name = "ALE/Pong-v5" # Wont render in ipynb

# env_name = "MountainCar-v0" # Example Discrete
# env_name = "MountainCarContinuous-v0" # Example Continuous


# env_name = "FrozenLake-v1"
try:
    register(
        id="FrozenLakeNoSlip-v1",
        entry_point="gymnasium.envs.toy_text.frozen_lake:FrozenLakeEnv",
        kwargs={"map_name": "4x4", "is_slippery": False},
        max_episode_steps=100,
        reward_threshold=0.78  # optimum = 0.74
    )
except:
    pass
env_name = "FrozenLakeNoSlip-v1"

render_mode_name = "human"
env = gym.make(env_name, render_mode = render_mode_name)


print("Observation Space: ", env.observation_space)
print("Action Space: ", env.action_space, " as type ", type(env.action_space))

Observation Space:  Discrete(16)
Action Space:  Discrete(4)  as type  <class 'gymnasium.spaces.discrete.Discrete'>


c:\Users\Reece S\Documents\GitHub\capstone4ds\.dqn_pong_env\Lib\site-packages\gymnasium\envs\registration.py:636: UserWarning: WARN: Overriding environment FrozenLakeNoSlip-v1 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")


Deep Q Model

In [9]:
class DeepQModel(nn.Module):
    def __init__(self, input_size, output_size):
        super(DeepQModel, self).__init__()

        print("Initializing DeepQModel")
        print("Input size:", input_size)
        print("Output size:", output_size)

        self.network = nn.Sequential(
            nn.Linear(input_size, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, output_size)
        )

    def forward(self, x):
        return self.network(x)


Define Agent

In [4]:
class Agent():
    def __init__(self, env):
        self.is_discrete = type(env.action_space) == gym.spaces.discrete.Discrete
        print("Is Discrete? ", self.is_discrete)

        if self.is_discrete:
            self.action_size = env.action_space.n
            print("Action size:", self.action_size)
        else:
            self.action_space_low = env.action_space.low
            self.action_space_high = env.action_space.high
            self.action_shape = env.action_space.shape
            print("Action range:", self.action_space_low, self.action_space_high)
    
    
    def get_action(self, observation):
        if self.is_discrete:
            action = random.choice(range(self.action_size))
        else:
            action = np.random.uniform(self.action_space_low, self.action_space_high, self.action_shape)

        return action


Define Tabular QAgent

In [13]:
class TabularQAgent(Agent):
    def __init__(self, env, discount_rate=0.97, learning_rate=0.1, epsilon=1.0):
        super().__init__(env)

        self.observation_size = env.observation_space.n
        print("Observation size:", self.observation_size)

        self.discount_rate = discount_rate
        self.learning_rate = learning_rate
        self.eps = epsilon

        self.build_model()

    def build_model(self):
        # Standard tabular initialization (zeros works best for FrozenLake)
        self.q_table = np.zeros((self.observation_size, self.action_size))

    def get_action(self, observation):
        if random.random() < self.eps:
            return super().get_action(observation)  # random action from parent
        else:
            return np.argmax(self.q_table[observation])

    def train(self, experience):
        observation, action, next_observation, reward, done = experience

        # Standard tabular Q-learning update
        q_current = self.q_table[observation, action]

        if done:
            q_target = reward
        else:
            q_target = reward + self.discount_rate * np.max(
                self.q_table[next_observation]
            )

        # Q update rule
        self.q_table[observation, action] += self.learning_rate * (
            q_target - q_current
        )

        # Decay epsilon at end of episode
        if done:
            self.eps *= 0.99


agent = TabularQAgent(env)


Is Discrete?  True
Action size: 4
Observation size: 16


Define Deep QAgent

In [14]:
class DeepQAgent(Agent):
    def __init__(self, env, discount_rate=0.97, learning_rate=0.001, epsilon=1.0):
        super().__init__(env)

        print("\n--- Initializing DeepQAgent ---")

        self.discount_rate = discount_rate
        self.learning_rate = learning_rate
        self.eps = epsilon

        print("Discount rate:", self.discount_rate)
        print("Learning rate:", self.learning_rate)
        print("Initial epsilon:", self.eps)

        # Detect state type
        if hasattr(env.observation_space, "n"):
            self.state_size = env.observation_space.n
            self.discrete_state = True
            print("State type: DISCRETE")
        else:
            self.state_size = env.observation_space.shape[0]
            self.discrete_state = False
            print("State type: CONTINUOUS")

        print("State size:", self.state_size)
        print("Action size:", self.action_size)

        # Build model
        self.model = DeepQModel(self.state_size, self.action_size)
        self.optimizer = optim.Adam(self.model.parameters(), lr=self.learning_rate)
        self.loss_fn = nn.MSELoss()

        print("--- DeepQAgent Ready ---\n")

    def process_state(self, state):
        if self.discrete_state:
            state_vector = np.zeros(self.state_size)
            state_vector[state] = 1.0
            print("One-hot encoded state:", state_vector)
            return torch.FloatTensor(state_vector)
        else:
            print("Raw continuous state:", state)
            return torch.FloatTensor(state)

        
    def get_action(self, observation):
        print("\nSelecting action...")
        print("Current epsilon:", self.eps)

        if random.random() < self.eps:
            action = super().get_action(observation)
            print("Random action selected:", action)
            return action

        state_tensor = self.process_state(observation)

        with torch.no_grad():
            q_values = self.model(state_tensor)

        print("Q-values:", q_values.numpy())

        action = torch.argmax(q_values).item()
        print("Greedy action selected:", action)

        return action

    
    def train(self, experience):
        observation, action, next_observation, reward, done = experience

        print("\n--- Training Step ---")
        print("State:", observation)
        print("Action:", action)
        print("Reward:", reward)
        print("Next State:", next_observation)
        print("Done:", done)

        state_tensor = self.process_state(observation)
        next_state_tensor = self.process_state(next_observation)

        # Current Q estimate
        q_values = self.model(state_tensor)
        q_value = q_values[action]

        print("Current Q-value:", q_value.item())

        # Compute target
        with torch.no_grad():
            next_q_values = self.model(next_state_tensor)
            max_next_q = torch.max(next_q_values)

            if done:
                target = torch.tensor(reward, dtype=torch.float32)
            else:
                target = reward + self.discount_rate * max_next_q

        print("Max next Q:", max_next_q.item())
        print("Target Q-value:", target.item())

        # Compute loss
        loss = self.loss_fn(q_value, target)

        print("TD Error:", (target - q_value).item())
        print("Loss:", loss.item())

        # Backpropagation
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        # Decay epsilon
        if done:
            self.eps *= 0.99
            print("Epsilon decayed to:", self.eps)

        print("--- End Training Step ---\n")


agent = DeepQAgent(env)

    


Is Discrete?  True
Action size: 4

--- Initializing DeepQAgent ---
Discount rate: 0.97
Learning rate: 0.001
Initial epsilon: 1.0
State type: DISCRETE
State size: 16
Action size: 4
Initializing DeepQModel
Input size: 16
Output size: 4
--- DeepQAgent Ready ---



Training Session

In [21]:

total_reward = 0

for ep in range(100):
    observation, info = env.reset()
    done = False
    while not done:
        action = agent.get_action(observation)
        next_observation, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        agent.train((observation, action, next_observation, reward, done))
        observation = next_observation
        total_reward += reward
        print("s:", observation, "a:", action)
        print("Episode: {}, Total reward: {}, Eps: {}".format(ep, total_reward, agent.eps))

        env.render()
        #print(agent.q_table)
    
        time.sleep(0.05)
        clear_output(wait=True)


Selecting action...
Current epsilon: 0.0024293023144556234
One-hot encoded state: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0.]
Q-values: [0.88229495 0.86860466 0.9977962  0.85518235]
Greedy action selected: 2

--- Training Step ---
State: 14
Action: 2
Reward: 1
Next State: 15
Done: True
One-hot encoded state: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0.]
One-hot encoded state: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]
Current Q-value: 0.9977961778640747
Max next Q: 0.7956266403198242
Target Q-value: 1.0
TD Error: 0.002203822135925293
Loss: 4.856832219957141e-06
Epsilon decayed to: 0.002405009291311067
--- End Training Step ---

s: 15 a: 2
Episode: 99, Total reward: 100, Eps: 0.002405009291311067
